# Part 2c — Risk-averse planning with CVaR

### When the average is the wrong thing to optimise

Part 2 minimised **expected** cost. That is the right objective if you will run this supply chain
many times and only the long-run average matters. For a critical-mineral chain the interesting
question is different: *how bad is bad?* A plan that is 2% cheaper on average and catastrophic in
the worst 5% of futures is not obviously better.

This notebook replaces the expectation with three alternatives and compares what each one builds.

| Objective | Optimises | Character |
|---|---|---|
| Risk-neutral | $\mathbb{E}[\text{cost}]$ | ignores the tail entirely |
| **CVaR($\alpha$)** | mean of the worst $\alpha$ fraction | tail-aware, still uses probabilities |
| Robust | $\max_k \text{cost}_k$ | one scenario dictates everything |
| **Hybrid** | $\lambda\,\mathbb{E} + (1-\lambda)\,\text{CVaR}$ | the one that usually wins |

### Why CVaR and not variance

Variance penalises upside and downside symmetrically, which is wrong for cost — being unexpectedly
cheap is not a risk. Value at Risk (VaR) picks out a quantile but says nothing about what lies
beyond it, and it is non-convex in the decision variables. **CVaR is the average of everything past
the quantile**, and — the reason it is usable at all — it has an exact linear representation.

### The Rockafellar–Uryasev trick

$$\text{CVaR}_\alpha = \min_{\eta}\;\; \eta + \frac{1}{\alpha}\,\mathbb{E}\big[(\text{cost}-\eta)^+\big]$$

Introduce $z_k \ge \text{cost}_k - \eta$, $z_k \ge 0$ and the whole thing is linear:

$$\text{CVaR}_\alpha \;=\; \min_{\eta, z}\;\; \eta + \frac{1}{\alpha}\sum_k p_k z_k$$

$\eta$ lands on the VaR at optimum without being constrained to. That is the entire method — two
variable families and two constraint families, and it composes with everything already in the model.

## 1. Setup and instance

In [ ]:
import os, random, itertools
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import matplotlib.pyplot as plt

for cand in ("gurobi.lic", os.path.join("..", "gurobi.lic")):
    if os.path.exists(cand):
        os.environ["GRB_LICENSE_FILE"] = os.path.abspath(cand); break
plt.rcParams.update({'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})

STAGES = ["MINE", "PROC", "MFG"]
REGS   = ["R1", "R2"]
NODES  = [(s, r) for s in STAGES for r in REGS]
ARCS   = [(s, r1, r2) for s in STAGES for r1 in REGS for r2 in REGS]
FIX  = {"MINE": 240., "PROC": 200., "MFG": 160.}
UNIT = {"MINE": 2.2,  "PROC": 2.9,  "MFG": 3.3}
OPC  = {"MINE": 0.8,  "PROC": 1.3,  "MFG": 1.6}
ETA  = {"MINE": 0.95, "PROC": 0.90, "MFG": 0.93}
CMIN, CMAX, PEN = 5.0, 70.0, 30.0
TAU = lambda r1, r2: 0.3 if r1 == r2 else 1.5

# R1 is the low-cost region - which is exactly why it is tempting to concentrate there.
REGION_COST = {"R1": 0.72, "R2": 1.00}

# Scenarios vary demand and a REGION-SPECIFIC cost shock.
#
# A shock that scales every region equally cannot change which plan is best - it just
# rescales the objective. For risk aversion to alter the *plan* rather than only the
# reported metric, the shock has to fall unevenly, so that concentrating capacity in one
# region is what carries the tail risk. R1 is cheap but occasionally disrupted; R2 is
# dearer and steady. That is the trade a risk-averse planner is being asked to make.
random.seed(7)
NK = 40
SC = []
for k in range(NK):
    hit = random.random() < 0.15                       # R1 disruption, ~15% of futures
    SC.append({"R1": 34.0*(0.6+0.9*random.random()),
               "R2": 22.0*(0.6+0.9*random.random()),
               "m_R1": 1.0 + (2.6*random.random() if hit else 0.0),
               "m_R2": 1.0 + 0.15*random.random()})
PK = [1.0/NK]*NK
nhit = sum(1 for s in SC if s["m_R1"] > 1.0)
print(f"{NK} scenarios | R1 disrupted in {nhit} ({100*nhit/NK:.0f}%)")
print(f"R1 multiplier range {min(s['m_R1'] for s in SC):.2f}-{max(s['m_R1'] for s in SC):.2f}")
print(f"R2 multiplier range {min(s['m_R2'] for s in SC):.2f}-{max(s['m_R2'] for s in SC):.2f}")


## 2. One model, four objectives

The second stage and the CVaR block are always present. Only the objective line changes, which is
what makes the comparison clean — every number below comes from the same feasible set.

In [ ]:
def solve(mode, alpha=0.10, lam=0.01, tiebreak=0.0, mipgap=0.0):
    m = gp.Model(); m.Params.OutputFlag = 0; m.Params.MIPGap = mipgap
    y = m.addVars(NODES, vtype=GRB.BINARY); c = m.addVars(NODES, lb=0.0, ub=CMAX)
    m.addConstrs((c[n] <= CMAX*y[n] for n in NODES))
    m.addConstrs((c[n] >= CMIN*y[n] for n in NODES))
    first = gp.quicksum(FIX[s]*y[s, r] + UNIT[s]*c[s, r] for (s, r) in NODES)

    x = m.addVars(NODES, range(NK), lb=0.0)
    f = m.addVars(ARCS,  range(NK), lb=0.0)
    u = m.addVars(REGS,  range(NK), lb=0.0)
    cost = {}
    for k in range(NK):
        m.addConstrs((x[s, r, k] <= c[s, r] for (s, r) in NODES))
        m.addConstrs((ETA[s]*x[s, r, k] == f.sum(s, r, "*", k) for (s, r) in NODES))
        for i, s in enumerate(STAGES):
            if i == 0: continue
            prev = STAGES[i-1]
            m.addConstrs((f.sum(prev, "*", r, k) == x[s, r, k] for r in REGS))
        m.addConstrs((f.sum("MFG", "*", r, k) + u[r, k] >= SC[k][r] for r in REGS))
        mk = {"R1": SC[k]["m_R1"], "R2": SC[k]["m_R2"]}
        cost[k] = (gp.quicksum(mk[r]*REGION_COST[r]*OPC[s]*x[s, r, k] for (s, r) in NODES)
                   + gp.quicksum(TAU(r1, r2)*f[s, r1, r2, k] for (s, r1, r2) in ARCS)
                   + gp.quicksum(PEN*u[r, k] for r in REGS))

    # --- Rockafellar-Uryasev CVaR block ---------------------------------
    eta = m.addVar(lb=-GRB.INFINITY, name="eta")
    z   = m.addVars(range(NK), lb=0.0, name="z")
    m.addConstrs((z[k] >= first + cost[k] - eta for k in range(NK)), name="cvar")
    CVaR = eta + (1.0/alpha)*gp.quicksum(PK[k]*z[k] for k in range(NK))
    EXP  = first + gp.quicksum(PK[k]*cost[k] for k in range(NK))

    if mode == "neutral":
        m.setObjective(EXP, GRB.MINIMIZE)
    elif mode == "cvar":
        # tiebreak>0 selects the lowest-mean plan among those that are CVaR-optimal
        m.setObjective(CVaR + tiebreak*EXP, GRB.MINIMIZE)
    elif mode == "robust":
        w = m.addVar(lb=0.0)
        m.addConstrs((w >= first + cost[k] for k in range(NK)))
        m.setObjective(w, GRB.MINIMIZE)
    elif mode == "hybrid":
        m.setObjective(lam*EXP + (1-lam)*CVaR, GRB.MINIMIZE)
    else:
        raise ValueError(mode)

    m.optimize()
    realised = sorted(first.getValue() + cost[k].getValue() for k in range(NK))
    ncrit = max(1, int(round(alpha*NK)))
    return dict(mode=mode,
                mean  = sum(realised)/NK,
                cvar  = sum(realised[-ncrit:])/ncrit,
                worst = realised[-1],
                capex = first.getValue(),
                cap   = {n: c[n].X for n in NODES},
                dist  = realised)

## 3. The comparison

In [ ]:
ALPHA, LAM = 0.10, 0.01
res = {mode: solve(mode, alpha=ALPHA, lam=LAM, tiebreak=1e-6)
       for mode in ["neutral", "cvar", "robust", "hybrid"]}

tab = pd.DataFrame([{"objective": r["mode"], "best": round(r["dist"][0], 1),
                     "mean": round(r["mean"], 1),
                     f"CVaR({ALPHA})": round(r["cvar"], 1), "worst": round(r["worst"], 1),
                     "spread": round(r["dist"][-1]-r["dist"][0], 1),
                     "capex": round(r["capex"], 1)} for r in res.values()])
print(tab.to_string(index=False))

# theory check: for ANY plan, mean <= CVaR <= worst
for r in res.values():
    assert r["mean"] <= r["cvar"] + 1e-6 <= r["worst"] + 1e-6, f"ordering violated for {r['mode']}"
print("\nOK - mean <= CVaR <= worst holds for every plan")


## 4. Why the hybrid wins — and it is not a compromise

The usual story is that a hybrid trades a little tail protection for a better average. On this
instance something sharper is happening, and it took a non-monotone $\lambda$ sweep to notice it.

**Pure CVaR is degenerate.** Many capacity plans achieve exactly the same CVaR, because the worst
$\alpha$ scenarios are dominated by shortfall in a region where several plans are equally bad. The
solver returns an arbitrary member of that tied set, so "the CVaR plan" is not well defined and its
mean cost is whatever the branch-and-bound happened to land on.

A tiny weight on the expectation is not buying a trade-off. It is **breaking a tie** — selecting,
among all CVaR-optimal plans, the one with the lowest average cost. That is why the hybrid can
improve the mean at *zero* cost in CVaR, which sounds impossible for a genuine trade-off and is
routine for a tie-break.

The cell below isolates it: same $\alpha$, same feasible set, the only difference is whether ties
are broken.

In [ ]:
raw  = solve("cvar", alpha=ALPHA, tiebreak=0.0)
tied = solve("cvar", alpha=ALPHA, tiebreak=1e-6)
print("pure CVaR, ties unbroken : CVaR %8.1f | mean %8.1f" % (raw["cvar"], raw["mean"]))
print("pure CVaR, ties broken   : CVaR %8.1f | mean %8.1f" % (tied["cvar"], tied["mean"]))
print("\nCVaR changed by %.4f   mean changed by %.1f"
      % (tied["cvar"]-raw["cvar"], tied["mean"]-raw["mean"]))
if abs(tied["cvar"]-raw["cvar"]) < 1e-6 and tied["mean"] < raw["mean"] - 1e-6:
    print("-> degenerate: the mean fell with CVaR unchanged, so those plans were tied")
else:
    print("-> not degenerate on this instance; the hybrid is a genuine trade-off here")


### Restated against a well-defined baseline

With ties broken, the comparison is honest: the hybrid is measured against the *best* pure-CVaR
plan, not an arbitrary one.

In [ ]:
n, cv, hy, rb = res["neutral"], res["cvar"], res["hybrid"], res["robust"]
print("hybrid vs pure CVaR")
print("  mean      %8.1f -> %8.1f   (%+.2f%%)" % (cv["mean"], hy["mean"], 100*(hy["mean"]-cv["mean"])/cv["mean"]))
print("  CVaR      %8.1f -> %8.1f   (%+.2f%%)" % (cv["cvar"], hy["cvar"], 100*(hy["cvar"]-cv["cvar"])/cv["cvar"]))
print("\nhybrid vs risk-neutral")
print("  mean      %8.1f -> %8.1f   (%+.2f%%)" % (n["mean"], hy["mean"], 100*(hy["mean"]-n["mean"])/n["mean"]))
print("  CVaR      %8.1f -> %8.1f   (%+.2f%%)" % (n["cvar"], hy["cvar"], 100*(hy["cvar"]-n["cvar"])/n["cvar"]))
print("  worst     %8.1f -> %8.1f   (%+.2f%%)" % (n["worst"], hy["worst"], 100*(hy["worst"]-n["worst"])/n["worst"]))
print("\nrobust")
print("  worst     %8.1f  (best possible)  but mean %8.1f (%+.2f%% vs risk-neutral)"
      % (rb["worst"], rb["mean"], 100*(rb["mean"]-n["mean"])/n["mean"]))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.2))
for k, r in res.items():
    ax[0].plot(r["dist"], label=k)
ax[0].set_xlabel("scenario (sorted)"); ax[0].set_ylabel("total cost")
ax[0].set_title("Cost distributions"); ax[0].legend()

ax[1].scatter([r["mean"] for r in res.values()], [r["cvar"] for r in res.values()], s=70)
for r in res.values():
    ax[1].annotate(r["mode"], (r["mean"], r["cvar"]), textcoords="offset points", xytext=(6, 5))
ax[1].set_xlabel("mean cost"); ax[1].set_ylabel(f"CVaR({ALPHA})")
ax[1].set_title("The trade-off plane - down and left is better")
plt.tight_layout(); plt.show()

## 5. Sensitivity to $\lambda$ and $\alpha$

In [ ]:
lams = [0.0, 0.005, 0.01, 0.05, 0.15, 0.4, 0.7, 1.0]
sweep = [dict(lam=l, **{k: v for k, v in solve("hybrid", alpha=ALPHA, lam=l,
                                               tiebreak=1e-9).items()
                        if k in ("mean", "cvar", "worst")}) for l in lams]
sw = pd.DataFrame(sweep)
print(sw.to_string(index=False, float_format=lambda v: f"{v:9.4g}"))
print("\nlambda = 0 leaves the tie unbroken; any lambda > 0 selects the")
print("lowest-mean plan among the CVaR-optimal set, with CVaR unchanged.")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sw["lam"], sw["mean"], "o-", label="mean")
ax.plot(sw["lam"], sw["cvar"], "s-", label=f"CVaR({ALPHA})")
ax.set_xlabel("$\\lambda$  (weight on the expectation)"); ax.set_ylabel("cost")
ax.set_title("$\\lambda=0$ is pure CVaR, $\\lambda=1$ is risk-neutral"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
alphas = [0.05, 0.10, 0.20, 0.35, 0.50]
rows = []
for a in alphas:
    r = solve("cvar", alpha=a)
    rows.append(dict(alpha=a, mean=round(r["mean"], 1), cvar=round(r["cvar"], 1),
                     worst=round(r["worst"], 1), capex=round(r["capex"], 1)))
print(pd.DataFrame(rows).to_string(index=False))
print("\nAs alpha -> 1 CVaR becomes the mean; as alpha -> 0 it approaches the worst case.")


## 6. What the risk-averse plan actually does differently

In [ ]:
capcmp = pd.DataFrame([{"node": f"{s}/{r}",
                        **{mode: round(res[mode]["cap"][s, r], 1) for mode in res}}
                       for (s, r) in NODES])
print(capcmp.to_string(index=False))
print("\ntotal capacity:")
for mode in res:
    print("  %-8s %6.1f" % (mode, sum(res[mode]["cap"].values())))

## 7. Carrying this into the production model

**It is additive.** The CVaR block is one scalar $\eta$, one $z_k$ per scenario, and one constraint
per scenario. It does not touch the network, the vintages, or the capacity logic — so it drops onto
the Part 5 core without restructuring anything.

**It composes with Part 2b.** The epigraph variable sits in the master; the subproblems are
unchanged. Risk aversion and decomposition are orthogonal.

**Choose $\alpha$ before you look at results.** $\alpha$ is a statement about which futures you are
willing to be unprepared for. Picking it after seeing the frontier is fitting the risk preference to
the answer.

**The tail must be populated to matter.** With symmetric well-behaved scenarios CVaR and the mean
often select the same plan and the whole apparatus is wasted — the same lesson as VSS in Part 2.
Check that the scenario set actually has a tail before paying for tail protection.

### Limitations here

- Randomness is in demand and a cost multiplier only. Disruptions — arcs disappearing — are the
  case Part 4f handles, and they are not smooth perturbations.
- Scenario probabilities are uniform and assumed known. Where they are contestable, the robust
  column is the honest one.